# Stage B — Align ERA5 to **Basin** Level (6-hour, Bhutan Local Time)

This stage takes the **merged ERA5 grid-level data** and the **grid→polygon mapping** generated in Stage A, and produces **basin×time** aggregates suitable for machine-learning feature engineering.

---

### **Inputs**

* **ERA5 Merged Parquet**

  * Local file: `data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet`
  * Automatic fallback: download from **Hugging Face** (`qlk0610/bhutan-climate`) if local file is missing
* **Grid→Polygon Mapping**

  * `data/spatial/grid_mapping/era5_grid_to_polygons.parquet`
  * Used to extract **basin\_id** only (watershed grouping ignored)

---

### **Outputs**

* **6-hour Aggregation:**
  `data/era5/era5_aligned/basin_level/era5_basin_6hour.parquet`
* **Daily Aggregation (optional):**
  `data/era5/era5_aligned/basin_level/era5_basin_daily.parquet`

---

### **Processing Notes**

* **Time Handling:** All timestamps are interpreted as **Bhutan local time (UTC+6)**, matching the merged ERA5 dataset.
* **Spatial Aggregation:** Basin values are computed as the **mean across grid cells** (with option to switch to area-weighted averaging later).
* **Coverage Columns:**

  * `n_cells` = number of contributing ERA5 grid cells (static per basin)
  * `low_coverage` = Boolean flag for insufficient grid coverage


### Requirements

In [1]:
#!pip3 install pandas pyarrow fastparquet huggingface_hub

In [2]:
from pathlib import Path
import os, subprocess
import pandas as pd
import numpy as np

### Resolve Project Root

In [3]:
def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p / "data").exists() and (p / "code").exists():
            return p
        if (p / ".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/qingfangliu/bhutan_climate_modeling


### Configuration (paths, HF fallback, options)

In [4]:
# --- Inputs ---
MAPPING_PARQUET = PROJECT_ROOT / "data/spatial/grid_mapping/era5_grid_to_polygons.parquet"

# ERA5 merged parquet (local preferred)
ERA5_LOCAL_PARQUET = PROJECT_ROOT / "data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet"

# Hugging Face fallback (if local missing)
USE_HF_IF_LOCAL_MISSING = True
HF_REPO_ID   = "qlk0610/bhutan-climate"
HF_REPO_FILE = "era5/era5_merged/merged_era5_6hour_1979_2025.parquet"

In [5]:
# --- Outputs ---
OUT_DIR_BASIN = PROJECT_ROOT / "data/era5/era5_aligned/basin_level"
OUT_6H        = OUT_DIR_BASIN / "era5_basin_6hour.parquet"
OUT_DAILY     = OUT_DIR_BASIN / "era5_basin_daily.parquet"

In [6]:
# --- Options ---
ROUND_COORDS_DECIMALS = 5   # round lat/lon before generating grid_id to avoid float quirks
DO_DAILY_ROLLUP       = True

### Resolve ERA5 source (local or Hugging Face)

In [7]:
ERA5_SOURCE_PATH = ERA5_LOCAL_PARQUET
if (not ERA5_SOURCE_PATH.exists()) and USE_HF_IF_LOCAL_MISSING:
    try:
        from huggingface_hub import hf_hub_download

        print("Local ERA5 parquet not found. Downloading from Hugging Face…")
        dl_path = hf_hub_download(
            repo_id=HF_REPO_ID,
            filename=HF_REPO_FILE,
            repo_type="dataset",     # <-- IMPORTANT: your repo is a DATASET
            revision="main",
            local_dir=PROJECT_ROOT / "data",
            local_dir_use_symlinks=False,
        )
        ERA5_SOURCE_PATH = Path(dl_path)
        print("HF downloaded to:", ERA5_SOURCE_PATH)
    except Exception as e:
        raise FileNotFoundError(f"ERA5 local missing and HF download failed: {e}")

print("Using ERA5 parquet:", ERA5_SOURCE_PATH)


Using ERA5 parquet: /Users/qingfangliu/bhutan_climate_modeling/data/era5/era5_merged/merged_era5_6hour_1979_2025.parquet


### Helper functions

In [8]:
def stable_grid_id(df_latlon: pd.DataFrame, round_decimals: int = None) -> pd.Series:
    """Stable ID by hashing (lat,lon). Optional rounding to avoid float drift."""
    xy = df_latlon[["latitude","longitude"]].copy()
    if round_decimals is not None:
        xy = xy.round({"latitude": round_decimals, "longitude": round_decimals})
    hashed = pd.util.hash_pandas_object(xy, index=False).astype("uint64")
    return "g" + hashed.map(lambda x: format(x, "016x"))

def detect_datetime_col(df: pd.DataFrame):
    for c in df.columns:
        lc = c.lower()
        if lc in ("datetime","date_time","time","timestamp"):
            return c
    # fallback: first datetime-like
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            return c
    raise ValueError("Could not find datetime column in ERA5 parquet.")


### Load mapping + ERA5; join via grid_id

In [9]:
# Load mapping (only basin-related columns; watershed ignored for now)
mp_cols = ["grid_id","latitude","longitude","basin_id","outside_flag"]
mp = pd.read_parquet(MAPPING_PARQUET, columns=mp_cols)

# Drop outside points
mp = mp.loc[~mp["outside_flag"] & mp["basin_id"].notna()].copy()
mp

,grid_id,latitude,longitude,basin_id,outside_flag
20,g545ff3c51071ec45,26.75,89.75,6.0,False
21,g02c74a9b46309376,26.75,90.00,8.0,False
32,g42f2b14eea3c6c5c,27.00,89.00,4.0,False
33,gde4936d2c33c4969,27.00,89.25,5.0,False
34,gad3704d82a24c393,27.00,89.50,5.0,False
...,...,...,...,...,...
117,gea01174e08082520,28.25,91.50,7.0,False
118,ge082077a1f32dbe0,28.25,91.75,7.0,False
119,g6ede76f1b14bfcb6,28.25,92.00,7.0,False
130,g464a1060d10869e7,28.50,91.00,7.0,False


In [10]:
# Load ERA5; read full (we’ll auto-detect dt col)
era = pd.read_parquet(ERA5_SOURCE_PATH)
dt_col = detect_datetime_col(era)
print("Detected datetime column:", dt_col)

# Build grid_id on ERA5 rows (after rounding to avoid float quirks)
era = era.copy()
era["latitude"] = era["latitude"].round(ROUND_COORDS_DECIMALS)
era["longitude"] = era["longitude"].round(ROUND_COORDS_DECIMALS)
era["grid_id"] = stable_grid_id(era[["latitude","longitude"]], round_decimals=None)
era

Detected datetime column: datetime


,latitude,longitude,datetime,temperature,dewpoint,wind_u,wind_v,potential_evaporation,runoff,snow_depth,snowmelt,soil_temperature,sub_surface_runoff,surface_runoff,solar_radiation,precipitation,grid_id
0,26.5,88.5,1979-01-01 05:30:00+05:30,8.779205,280.687500,1.184982,-0.662811,1.918059e-06,0.000015,0.0,0.0,285.450195,0.000015,0.0,128.0,0.0,gf81d7fb303616195
1,26.5,88.5,1979-01-01 11:30:00+05:30,23.847076,282.582336,-0.500717,-0.115112,-5.577810e-04,0.000016,0.0,0.0,294.535645,0.000016,0.0,2433472.0,0.0,gf81d7fb303616195
2,26.5,88.5,1979-01-01 17:30:00+05:30,20.769440,282.469971,0.937103,-0.837021,-6.991206e-06,0.000016,0.0,0.0,293.682373,0.000016,0.0,0.0,0.0,gf81d7fb303616195
3,26.5,88.5,1979-01-01 23:30:00+05:30,11.567841,281.223572,0.596100,-1.193222,-2.800487e-06,0.000016,0.0,0.0,287.762207,0.000016,0.0,0.0,0.0,gf81d7fb303616195
4,26.5,88.5,1979-01-02 05:30:00+05:30,9.451782,281.199463,0.905502,-1.021103,3.932510e-07,0.000016,0.0,0.0,285.927246,0.000016,0.0,64.0,0.0,gf81d7fb303616195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9183640,28.5,92.0,2025-07-23 18:00:00+06:00,NaN,NaN,NaN,NaN,NaN,0.000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,g3f9cf585266c6628
9183641,28.5,92.0,2025-07-24 00:00:00+06:00,NaN,NaN,NaN,NaN,NaN,0.000005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,g3f9cf585266c6628
9183642,28.5,92.0,2025-07-24 06:00:00+06:00,NaN,NaN,NaN,NaN,NaN,0.000006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,g3f9cf585266c6628
9183643,28.5,92.0,2025-07-24 12:00:00+06:00,NaN,NaN,NaN,NaN,NaN,0.000019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,g3f9cf585266c6628


In [11]:
# Join to get basin_id
era = era.merge(mp[["grid_id","basin_id"]], on="grid_id", how="left")

before = len(era)
era = era.loc[era["basin_id"].notna()].copy()
after = len(era)
print(f"Joined rows with basin_id: {after:,} / {before:,} ({after/before:.1%})")

# Identify variable columns (numeric, excluding coords/ids)
exclude = {"latitude","longitude","grid_id",dt_col,"basin_id","outside_flag"}
num_cols = [c for c in era.columns if c not in exclude and pd.api.types.is_numeric_dtype(era[c])]
print("Variables to aggregate (sample):", num_cols[:10], " ... total:", len(num_cols))

Joined rows with basin_id: 5,033,998 / 9,183,645 (54.8%)
Variables to aggregate (sample): ['temperature', 'dewpoint', 'wind_u', 'wind_v', 'potential_evaporation', 'runoff', 'snow_depth', 'snowmelt', 'soil_temperature', 'sub_surface_runoff']  ... total: 13


### Aggregate to basin × time (6-hour areal averages)

In [12]:
# Ensure datetime dtype
era[dt_col] = pd.to_datetime(era[dt_col])

# Aggregate: mean across grid cells per (basin_id, time)
group_keys = ["basin_id", dt_col]
agg_dict = {c: "mean" for c in num_cols}

basin_6h = (era.groupby(group_keys, as_index=False)
              .agg(agg_dict)
              .sort_values(group_keys)
              .reset_index(drop=True))

# Attach static coverage per basin (n_cells from mapping)
n_cells_per_basin = (mp.groupby("basin_id")["grid_id"].nunique()
                       .rename("n_cells").reset_index())
basin_6h = basin_6h.merge(n_cells_per_basin, on="basin_id", how="left")
basin_6h["low_coverage"] = (basin_6h["n_cells"] <= 1)

print("Rows:", len(basin_6h), "| Basins:", basin_6h["basin_id"].nunique())

Rows: 612243 | Basins: 9


In [13]:
print(basin_6h.head(10))

   basin_id                  datetime  temperature    dewpoint    wind_u  \
0       1.0 1979-01-01 05:30:00+05:30     5.002838  275.337891  0.091721   
1       1.0 1979-01-01 11:30:00+05:30    15.956451  279.131165  0.099380   
2       1.0 1979-01-01 17:30:00+05:30    10.506744  280.019775  0.142670   
3       1.0 1979-01-01 23:30:00+05:30     6.159637  277.942322  0.047760   
4       1.0 1979-01-02 05:30:00+05:30     6.106079  276.787354  0.025620   
5       1.0 1979-01-02 11:30:00+05:30    15.736481  277.508026 -0.090744   
6       1.0 1979-01-02 17:30:00+05:30    10.292145  279.303467  0.111740   
7       1.0 1979-01-02 23:30:00+05:30     5.082703  276.808990  0.037186   
8       1.0 1979-01-03 05:30:00+05:30     4.457123  274.602539  0.015549   
9       1.0 1979-01-03 11:30:00+05:30    15.492340  278.916870  0.001831   

     wind_v  potential_evaporation    runoff  snow_depth  snowmelt  \
0 -1.507050          -1.419801e-06  0.000028         0.0       0.0   
1  1.462524          -3

In [14]:
print(n_cells_per_basin)

   basin_id  n_cells
0       1.0        2
1       3.0       11
2       4.0        1
3       5.0        5
4       6.0        8
5       7.0       28
6       8.0       14
7       9.0        3
8      10.0        2


### Daily roll-up (Bhutan local days)

This step builds an **aggregation specification (`agg_spec`)** to roll up 6-hourly data into daily basin values:

| **Column Type**      | **Examples**                                  | **Aggregation Rule** | **Rationale**                                   |
|---------------------|----------------------------------------------|---------------------|-----------------------------------------------|
| **Flux-like variables** | `precipitation`, `runoff`, `solar_radiation`, `snowmelt` | **sum**             | Daily total over all 6-hour windows            |
| **State-like variables** | `temperature`, `soil moisture`, `dewpoint`   | **mean**            | Daily mean representative of entire day        |
| **Coverage – n_cells**  | (static per basin)                           | **max**             | Retain cell count (constant within basin)      |
| **Coverage – low_coverage** | (boolean flag)                              | **max**             | Logical OR: flag day if **any** slot was low   |

In [15]:
basin_daily = None
if DO_DAILY_ROLLUP:
    # Treat these as flux-like (sum daily). Adjust to match your variable names.
    flux_like = {
        "precipitation","runoff","surface_runoff","sub_surface_runoff",
        "snowmelt","potential_evaporation","solar_radiation"
    }
    keep_cols = ["basin_id", dt_col, "n_cells", "low_coverage"] + num_cols
    tmp = basin_6h[keep_cols].copy()
    
    # Robust local-date extraction (handles tz-aware timestamps + the 1987-10-01 jump)
    LOCAL_TZ = "Asia/Thimphu"
    dt_s = tmp[dt_col]

    if getattr(dt_s.dt, "tz", None) is not None:
        # tz-aware → convert to local, drop tz, then take calendar date
        dt_local_naive = dt_s.dt.tz_convert(LOCAL_TZ).dt.tz_localize(None)
    else:
        # already naive local → use as-is
        dt_local_naive = dt_s
    tmp["date_local"] = dt_local_naive.dt.date

    agg_spec = {}
    for c in num_cols:
        agg_spec[c] = ("sum" if c in flux_like else "mean")
    # Coverage flags: keep static n_cells; 'low_coverage' = any true in day
    agg_spec["n_cells"] = "max"
    agg_spec["low_coverage"] = "max"

    basin_daily = (tmp.groupby(["basin_id","date_local"], as_index=False)
                     .agg(agg_spec)
                     .sort_values(["basin_id","date_local"])
                     .reset_index(drop=True))
    print("Daily rows:", len(basin_daily))


Daily rows: 153063


In [16]:
print(basin_daily.head(10))

   basin_id  date_local  temperature    dewpoint    wind_u    wind_v  \
0       1.0  1979-01-01     9.406418  278.107788  0.095383 -0.336563   
1       1.0  1979-01-02     9.304352  277.601959  0.020950 -0.234264   
2       1.0  1979-01-03     9.403366  278.130890  0.045818 -0.271885   
3       1.0  1979-01-04     8.499245  276.883438  0.004963  0.078732   
4       1.0  1979-01-05     8.388046  276.046249  0.166935  0.024647   
5       1.0  1979-01-06     9.662666  277.489319  0.148655 -0.357235   
6       1.0  1979-01-07    10.579720  278.112221  0.024178 -0.220642   
7       1.0  1979-01-08     9.662506  278.576973  0.155849  0.026913   
8       1.0  1979-01-09     9.879608  279.110481  0.079041 -0.194466   
9       1.0  1979-01-10    10.291290  279.255226 -0.012028 -0.048607   

   potential_evaporation    runoff  snow_depth  snowmelt  soil_temperature  \
0              -0.000307  0.000108         0.0       0.0        281.447205   
1              -0.000304  0.000104         0.0     

### QA checks (temporal completeness & hours)

In [17]:
# Check hour set (should be subset of {0,6,12,18} local)
hours = basin_6h[dt_col].dt.hour.unique()
print("Unique hours present:", sorted(hours.tolist()))

Unique hours present: [0, 5, 6, 11, 12, 17, 18, 23]


In [18]:
# Quick missing-stamp diagnostic per basin (6h cadence expected)
def expected_count(dts):
    # approximate: compute per-basin span / 6h + 1
    span_h = (dts.max() - dts.min()) / pd.Timedelta(hours=1)
    return int(span_h // 6) + 1

sample = (basin_6h.groupby("basin_id")[dt_col]
            .agg(["min","max","count"])
            .rename(columns={"count":"observed"})
         )
sample["expected_approx"] = sample.apply(lambda r: expected_count(pd.Series([r["min"], r["max"]])), axis=1)
sample["obs/exp_%"] = (sample["observed"] / sample["expected_approx"] * 100).round(1)
print(sample.head(10))

                               min                       max  observed  \
basin_id                                                                 
1.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
3.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
4.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
5.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
6.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
7.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
8.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
9.0      1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   
10.0     1979-01-01 05:30:00+05:30 2025-07-24 18:00:00+06:00     68027   

          expected_approx  obs/exp_%  
basin_id                              
1.0                 68027      100.0  
3.0                 68027      100.0  
4.0                 68027    

### Save outputs

In [19]:
OUT_DIR_BASIN.mkdir(parents=True, exist_ok=True)
basin_6h.to_parquet(OUT_6H, index=False)
print("Saved 6-hour basin file:", OUT_6H)

if DO_DAILY_ROLLUP and basin_daily is not None:
    basin_daily.to_parquet(OUT_DAILY, index=False)
    print("Saved daily basin file:", OUT_DAILY)


Saved 6-hour basin file: /Users/qingfangliu/bhutan_climate_modeling/data/era5/era5_aligned/basin_level/era5_basin_6hour.parquet
Saved daily basin file: /Users/qingfangliu/bhutan_climate_modeling/data/era5/era5_aligned/basin_level/era5_basin_daily.parquet
